<a href="https://colab.research.google.com/github/VJ13SS/Agentic_AI_Code_Generator/blob/main/LogClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/synthetic_logs.csv")

In [ ]:
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [ ]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [ ]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

##Clustering using DBSCAN (Density Based Clustering)

DBSCAN helps in identifying nested clusters if any which is not possible for k means

Clusters are in high density regions and outliers are in low density regions

For each point we choose other points similar to that point..
A circle is drawn around the core point with the radius (epsilon).

Here for each log we generate the embedding (text vector in n dimensional space).

We define core points as the one which is close to min_points on the circle.

After finding the all the core points we randomly pick a point and assign it to a Cluster and all the other points which wraps around the circle of the core point belongs to that Cluster and is extended further.

All non core points which are close to a core points of a cluster are assigned to that Cluster.Non - core points are not used to extend the cluster.

Any non core points which are not close to any of the cluster are considered to be outliers.

epsilon (hyper parameter)  which is the radius of the circle.

If two cluster share a common point(like they lies on the border of both the cluster) those points are added to the cluster it reaches first..
And the border point is not used to further expand the cluster.


Here cosine distance is used to check similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

SentenceTransformer is used for sentence vectorization,to generate embeddings

(It uses BERT)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(df['log_message'].tolist())

In [ ]:
embeddings[:2]

array([[-1.02939598e-01,  3.35459672e-02, -2.20261216e-02,
         1.55101111e-03, -9.86925978e-03, -1.78956240e-01,
        -6.34409934e-02, -6.01761788e-02,  2.81109102e-02,
         5.99620752e-02, -1.72618423e-02,  1.43358926e-03,
        -1.49560034e-01,  3.15287360e-03, -5.66031113e-02,
         2.71685515e-02, -1.49890641e-02, -3.54037993e-02,
        -3.62936519e-02, -1.45410383e-02, -5.61495451e-03,
         8.75539407e-02,  4.55120020e-02,  2.50964053e-02,
         1.00187557e-02,  1.24267498e-02, -1.39923573e-01,
         7.68696293e-02,  3.14095095e-02, -4.15246794e-03,
         4.36902344e-02,  1.71250012e-02, -8.00950974e-02,
         5.74006066e-02,  1.89092178e-02,  8.55262130e-02,
         3.96398790e-02, -1.34371787e-01, -1.44360214e-03,
         3.06703593e-03,  1.76854044e-01,  4.44889534e-03,
        -1.69274993e-02,  2.24266425e-02, -4.35049236e-02,
         6.09034114e-03, -9.98167321e-03, -6.23973198e-02,
         1.07372757e-02, -6.04894431e-03, -7.14660957e-0

In [ ]:
#DBSCAN

dbscan = DBSCAN(eps=0.2, min_samples=1, metric='cosine')

In [ ]:
clusters = dbscan.fit_predict(embeddings)

In [ ]:
df["cluster"] = clusters

In [ ]:
df.head(3)

,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2


In [ ]:
df[df.cluster == 1]

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,bert,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,bert,1
450,10/27/2025 5:59,ThirdPartyAPI,Email delivery system encountered an error,Error,bert,1
477,12/2/2025 10:30,AnalyticsEngine,Email transmission error caused service impact,Critical Error,bert,1
570,11/7/2025 18:08,ThirdPartyAPI,Email service impacted by sending failure,Critical Error,bert,1
678,4/28/2025 15:13,AnalyticsEngine,Email delivery problem affected system,Critical Error,bert,1


In [ ]:
cluster_counts = df['cluster'].value_counts()

In [ ]:
cluster_counts

,count
cluster,
0,1017
5,147
11,100
13,86
7,60
...,...
131,1
132,1
133,1


In [ ]:
large_clusters = cluster_counts[cluster_counts > 10].index

In [ ]:
#Clusters with number of elements greater than 10

large_clusters

Index([ 0,  5, 11, 13,  7,  8, 21,  3,  4, 17,  6, 32, 16, 20,  9,  1, 10, 34,
       53, 14, 52, 18, 42, 25, 59, 26],
      dtype='int64', name='cluster')

In [ ]:
for cluster in large_clusters:
  print(f"Cluster {cluster}: ")
  print(df[df["cluster"] == cluster]["log_message"].head(5).to_string(index = False))
  print()

Cluster 0: 
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...

Cluster 5: 
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...

Cluster 11: 
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.

Cluster 13: 
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.

Cluster 7: 
Multiple bad login attempts detected on user 85...
Multiple login failures occurred on user 9052 a...
  

We can see that Cluster 10,9,16,32,11,5 ... follows a pattern which can be identified using Regular expression

In [ ]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message,re.IGNORECASE):
            return label
    return None

In [ ]:
classify_with_regex("User uSEr123 logged in.")

'User Action'

In [ ]:
classify_with_regex("Hello")

In [ ]:
df["regex_label"] = df["log_message"].apply(classify_with_regex)

In [ ]:
df[df["regex_label"].notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10/11/2025 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,1/4/2025 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,11,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,21,System Notification
2381,9/5/2025 6:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2394,4/3/2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2395,5/2/2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,13,System Notification


In [ ]:
df[df["regex_label"].notnull()].shape

#they we classified correctly by regex

(500, 7)

In [ ]:
df_non_regex = df[df["regex_label"].isnull()].copy()

df_non_regex.shape

(1910, 7)

For non regex we need to use BERT or llm

In [ ]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 5].index.tolist())

['Workflow Error', 'Deprecation Warning']


We had found that legacy CRM source has only few samples...so we classify it using llm

In [ ]:
df_non_regex[df_non_regex.source == "LegacyCRM"]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
60,2025-10-06 16:55:23,LegacyCRM,Lead conversion failed for prospect ID 7842 du...,Workflow Error,llm,24,None
255,2025-05-03 16:55:35,LegacyCRM,API endpoint 'getCustomerDetails' is deprecate...,Deprecation Warning,llm,48,None
377,2025-06-24 12:16:29,LegacyCRM,Customer follow-up process for lead ID 5621 fa...,Workflow Error,llm,62,None
1325,2025-04-17 07:33:44,LegacyCRM,Escalation rule execution failed for ticket ID...,Workflow Error,llm,105,None
1734,2025-04-30 07:47:30,LegacyCRM,The 'ExportToCSV' feature is outdated. Please ...,Deprecation Warning,llm,118,None
1826,2025-01-23 10:33:36,LegacyCRM,Support for legacy authentication methods will...,Deprecation Warning,llm,122,None
2217,2025-05-12 09:46:54,LegacyCRM,Task assignment for TeamID 3425 could not comp...,Workflow Error,llm,133,None


In [ ]:
df_non_legacy = df_non_regex[df_non_regex.source != "LegacyCRM"]

df_non_legacy.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [ ]:
df_non_legacy

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None
...,...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,None
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,7,None
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,None
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,None


In [ ]:
filtered_embeddings = model.encode(df_non_legacy["log_message"].tolist())

filtered_embeddings[:2]

array([[-1.02939598e-01,  3.35459672e-02, -2.20261216e-02,
         1.55101111e-03, -9.86925978e-03, -1.78956240e-01,
        -6.34409934e-02, -6.01761788e-02,  2.81109102e-02,
         5.99620752e-02, -1.72618423e-02,  1.43358926e-03,
        -1.49560034e-01,  3.15287360e-03, -5.66031113e-02,
         2.71685515e-02, -1.49890641e-02, -3.54037993e-02,
        -3.62936519e-02, -1.45410383e-02, -5.61495451e-03,
         8.75539407e-02,  4.55120020e-02,  2.50964053e-02,
         1.00187557e-02,  1.24267498e-02, -1.39923573e-01,
         7.68696293e-02,  3.14095095e-02, -4.15246794e-03,
         4.36902344e-02,  1.71250012e-02, -8.00950974e-02,
         5.74006066e-02,  1.89092178e-02,  8.55262130e-02,
         3.96398790e-02, -1.34371787e-01, -1.44360214e-03,
         3.06703593e-03,  1.76854044e-01,  4.44889534e-03,
        -1.69274993e-02,  2.24266425e-02, -4.35049236e-02,
         6.09034114e-03, -9.98167321e-03, -6.23973198e-02,
         1.07372757e-02, -6.04894431e-03, -7.14660957e-0

Treating the filtered_embeddings as X and target_label as y we are training a logistic regression model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
X = filtered_embeddings
y = df_non_legacy['target_label']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.3,random_state = 42)

In [ ]:
clf = LogisticRegression()

In [ ]:
clf.fit(X_train,y_train)

LogisticRegression()

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
report = classification_report(y_test,y_pred)

print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [ ]:
import joblib

In [ ]:
joblib.dump(clf, 'log_classification_model.pkl')

In [ ]:
from google.colab import files
files.download('log_classification_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
temp = joblib.load("/content/log_classification_model.pkl")

In [ ]:
test_sen ="Email service affected by failed transmission"

test_sen = model.encode(test_sen).tolist()

In [ ]:
temp.predict([test_sen])

array(['Critical Error'], dtype=object)

#RegEx

For logs which follows the default..it is better to classify them using regex which is fast and accurate.

In [ ]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User (User\d+|.*) logged (in|out).*": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.*": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message,re.IGNORECASE):
            return label
    return None

#BERT

For logs which cannot be classified using regex..we check if the source has enough samples...we had found that there are only few samples with source of LegacyCRM this all the other samples are classified using the logistic regression classifier

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

In [ ]:
import joblib

In [ ]:
transformer_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
classification_model = joblib.load("/content/log_classification_model.pkl")

In [ ]:
def classify_with_bert(log_message):
  message_embedding = transformer_model.encode(log_message)
  probabilities = classification_model.predict_proba([message_embedding])[0]

  if max(probabilities) < 0.5:
    return "UnClassified"
  predicted_label = classification_model.predict([message_embedding])[0]

  return predicted_label

In [ ]:
classify_with_bert("Multiple login failures occurred on user 6454 account")

'Security Alert'

In [ ]:
classify_with_bert("Hey bro, Chill ya!")

'UnClassified'

Logistic Regression works on probabilities..
It classifies to the one which got the highest probability

classify_with_bert("Hey bro, Chill ya!") here we got output class as "Security Alert" and the probabilities is [0.34736008, 0.14754995, 0.05375047, 0.05621691, 0.39512259] which is poor which means that the model is confused.

So we are updating our function such that if the maximum probability is greater than a threshold we treat it as a member of the class else UnClassified

#LLM

For the logs with source LegacyCRM we will classify using LLM (using the llama 3 model

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your api key"

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.1 MB/s eta 0:00:00


In [ ]:
from groq import Groq

In [ ]:
from groq import Groq

client = Groq()
prompt = "What is the difference between AI and ML?"
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
)

print(completion.choices[0].message.content)

Artificial Intelligence (AI) and Machine Learning (ML) are two closely related but distinct concepts in the field of computer science. While they are often used interchangeably, there is a subtle difference between them.

**Artificial Intelligence (AI):**
AI refers to the broader field of research and development aimed at creating machines that can perform tasks that typically require human intelligence. This includes a wide range of disciplines such as:

* Reasoning and problem-solving
* Knowledge representation
* Planning and decision-making
* Natural Language Processing (NLP)
* Computer vision
* Robotics

The primary goal of AI is to create systems that can think and act like humans, with the ability to learn, reason, and apply knowledge to solve complex problems.

**Machine Learning (ML):**
ML, on the other hand, is a subset of AI that focuses specifically on the development of algorithms and statistical models that enable machines to learn from data, without being explicitly progr

In [ ]:
def classify_with_llm(log_message):
  client = Groq()
  prompt = f"""Classify the log message into one of these categories:
  (1) Workflow Error, (2) Deprication Warning.
  If you can't figure out a category, return UnClassified".
  Only return the category name. No preamble.
  Log message : {log_message}"""

  completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
  )

  return completion.choices[0].message.content

In [ ]:
classify_with_llm("Hey bro, chill ya")

'UnClassified'

In [ ]:
classify_with_llm("API endpoint 'getCustomerDetails' is deprecated")

'Deprication Warning'

#Classify

In [ ]:
def classify(logs):
  labels = []
  for source,log_msg in logs:
    label = classify_log(source,log_msg)
    labels.append(label)
  return labels

In [ ]:
def classify_log(source,log_msg):
  if source == "LegacyCRM":
    label = classify_with_llm(log_msg)
  else:
    label = classify_with_regex(log_msg)
    if label is None:
      label = classify_with_bert(log_msg)

  return label

In [ ]:
input_path = "/content/test.csv"

In [ ]:
import pandas as pd

def classify_csv(input_file):
  df = pd.read_csv(input_file)
  df["target_label"] = classify(list(zip(df["source"],df["log_message"])))

  output_file = "output.csv"
  df.to_csv(output_file,index = False)

In [ ]:
#df["target_label"] = classify(list(zip(df["source"],df["log_message"])))


#this takes the two columns (source,log_message) from the data frame, the zip function converts them to row wise pairs (tuples) then converts to list of tuples

In [ ]:
classify_csv(input_path)

In [ ]:
logs = [
     ("ModernCRM", "IP 192.168.133.114 blocked due to potential attack"),
     ("BillingSystem", "User 12345 logged in."),
     ("AnalyticsEngine", "File data_6957.csv uploaded successfully by user User265."),
     ("AnalyticsEngine", "Backup completed successfully."),
    ("ModernHR", "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1 RCODE  200 len: 1583 time: 0.1878400"),
     ("ModernHR", "Admin access escalation detected for user 9429"),
     ("LegacyCRM", "Case escalation for ticket ID 7324 failed because the assigned support agent is no longer active."),
     ("LegacyCRM", "Invoice generation process aborted for order ID 8910 due to invalid tax calculation module."),
     ("LegacyCRM", "The 'BulkEmailSender' feature is no longer supported. Use 'EmailCampaignManager' for improved functionality."),
     ("LegacyCRM", " The 'ReportGenerator' module will be retired in version 4.0. Please migrate to the 'AdvancedAnalyticsSuite' by Dec 2025")
    ]
labels = classify(logs)

labels

['Security Alert',
 'User Action',
 'System Notification',
 'System Notification',
 'HTTP Status',
 'Security Alert',
 'Workflow Error',
 'Workflow Error',
 'Deprication Warning',
 'Deprication Warning']

In [ ]:
for i in range(len(labels)):
  log = logs[i]
  label = labels[i]

  print(f"Source : {log[0]}")
  print(f"Log Message : {log[1]}")
  print(f"Classified Class : {label}")
  print("\n")

Source : ModernCRM
Log Message : IP 192.168.133.114 blocked due to potential attack
Classified Class : Security Alert


Source : BillingSystem
Log Message : User 12345 logged in.
Classified Class : User Action


Source : AnalyticsEngine
Log Message : File data_6957.csv uploaded successfully by user User265.
Classified Class : System Notification


Source : AnalyticsEngine
Log Message : Backup completed successfully.
Classified Class : System Notification


Source : ModernHR
Log Message : GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1 RCODE  200 len: 1583 time: 0.1878400
Classified Class : HTTP Status


Source : ModernHR
Log Message : Admin access escalation detected for user 9429
Classified Class : Security Alert


Source : LegacyCRM
Log Message : Case escalation for ticket ID 7324 failed because the assigned support agent is no longer active.
Classified Class : Workflow Error


Source : LegacyCRM
Log Message : Invoice generation process aborted for order ID 8910 due 